# Pokemon Card Price Finder
Foto hochladen → Claude Vision identifiziert die Karte → Pokemon TCG API liefert Marktpreise (Cardmarket EUR + TCGPlayer USD)

In [ ]:
# Setup
import anthropic
import requests
import base64
import json
import os
from pathlib import Path
from IPython.display import display, HTML

# .env laden (ANTHROPIC_API_KEY + optional POKEMON_TCG_API_KEY)
for _p in [Path(".env"), Path("../../.env"), Path("../../../.env")]:
    if _p.exists():
        for _line in _p.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break

POKEMON_TCG_API_KEY = os.getenv("POKEMON_TCG_API_KEY", "")

In [ ]:
def load_image(path: str) -> tuple[str, str]:
    ext = Path(path).suffix.lower()
    media_types = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    return data, media_types.get(ext, "image/jpeg")


def normalize_name(name: str) -> list[str]:
    """Gibt Namensvarianten zurück: API nutzt Bindestrich vor GX/EX/V/VMAX/VSTAR."""
    variants = [name]
    for suffix in [" GX", " EX", " V", " VMAX", " VSTAR", " VStar"]:
        if name.endswith(suffix):
            base = name[: -len(suffix)]
            hyphenated = base + suffix.replace(" ", "-")
            if hyphenated not in variants:
                variants.append(hyphenated)
    return variants


def identify_card(image_path: str) -> dict:
    client = anthropic.Anthropic()
    image_data, media_type = load_image(image_path)
    prompt = """Analyze this Pokemon card photo and return ONLY a JSON object:
{
  "name": "exact name on card (e.g. Charizard, Gengar & Mimikyu GX, Pikachu V)",
  "set_name": "set name or null",
  "card_number": "full number like 53/181 or null",
  "set_number": "number before slash as string or null",
  "rarity": "Rare Holo GX / Rare Ultra / Rare Rainbow / Common / etc.",
  "condition_estimate": "Near Mint/Lightly Played/Moderately Played/Heavily Played/Damaged",
  "is_holo": true or false,
  "is_first_edition": true or false,
  "is_shadowless": true or false,
  "language": "English/German/Japanese/etc.",
  "card_type": "Pokemon/Trainer/Energy"
}
Return ONLY JSON. Use null (not the string null) for unknown values."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_data}},
            {"type": "text", "text": prompt},
        ]}],
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(raw)


def lookup_prices(card_info: dict) -> list[dict]:
    headers = {"X-Api-Key": POKEMON_TCG_API_KEY} if POKEMON_TCG_API_KEY else {}
    name = card_info.get("name") or ""
    set_name = card_info.get("set_name") or ""
    set_num = str(card_info.get("set_number") or "")

    cards = []
    for name_variant in normalize_name(name):
        query = f'name:"{name_variant}"'
        if set_name:
            query += f' set.name:"{set_name}"'
        resp = requests.get(
            "https://api.pokemontcg.io/v2/cards",
            headers=headers,
            params={"q": query, "select": "id,name,set,number,rarity,tcgplayer,cardmarket", "pageSize": 10},
            timeout=15,
        )
        resp.raise_for_status()
        cards = resp.json().get("data", [])
        if cards:
            break

    # Fallback: name only (no set filter)
    if not cards:
        for name_variant in normalize_name(name):
            resp = requests.get(
                "https://api.pokemontcg.io/v2/cards",
                headers=headers,
                params={"q": f'name:"{name_variant}"', "select": "id,name,set,number,rarity,tcgplayer,cardmarket", "pageSize": 10},
                timeout=15,
            )
            resp.raise_for_status()
            cards = resp.json().get("data", [])
            if cards:
                break

    # Narrow by card number if unambiguous
    if set_num and len(cards) > 1:
        filtered = [c for c in cards if c.get("number") == set_num]
        if filtered:
            cards = filtered

    return cards


def fmt(val, prefix=""):
    return f"{prefix}{val}" if val is not None else "--"


def display_results(card_info: dict, cards: list[dict]):
    SEP = "=" * 64
    sep = "-" * 64

    print(SEP)
    print("  POKEMON CARD PRICE FINDER")
    print(SEP)
    print(f"  Karte:      {card_info.get('name') or '--'}")
    print(f"  Set:        {card_info.get('set_name') or '(unbekannt)'}")
    print(f"  Nr:         {card_info.get('card_number') or '--'}")
    print(f"  Seltenheit: {card_info.get('rarity') or '--'}")
    print(f"  Zustand:    {card_info.get('condition_estimate') or '--'}")
    print(f"  Sprache:    {card_info.get('language') or '--'}  |  Holo: {card_info.get('is_holo')}  |  1st Ed: {card_info.get('is_first_edition')}  |  Shadowless: {card_info.get('is_shadowless')}")

    if not cards:
        print(f"\n  Kein Treffer in der Pokemon TCG Datenbank.")
        print("  Tipp: Kartennummer pruefen und MANUAL_CARD_ID in letzter Zelle setzen.")
        return

    if len(cards) > 1:
        print(f"\n  {len(cards)} Varianten gefunden -- Kartennummer pruefen!")
        print(f"  {'ID':<14} {'Nr':>6}  {'Rarity':<28}  TCGPlayer Market  CM Trend")
        print(f"  {sep}")
        for c in cards:
            tcg_m = None
            for pd in (c.get("tcgplayer") or {}).get("prices", {}).values():
                tcg_m = pd.get("market"); break
            cm_t = ((c.get("cardmarket") or {}).get("prices") or {}).get("trendPrice")
            print(f"  {c['id']:<14} {c.get('number',''):>6}  {c.get('rarity',''):.<28}  {fmt(tcg_m,'$'):>16}  {fmt(cm_t,'EUR'):>10}")
        print(f"\n  Details fuer alle Varianten:")

    for i, card in enumerate(cards, 1):
        c_set = card.get("set", {})
        print(f"\n  [{i}] {card.get('name')} | {c_set.get('name')} | Nr. {card.get('number')} | {card.get('rarity')} | {card.get('id')}")
        print(f"  {sep}")

        tcg = card.get("tcgplayer", {})
        if tcg and tcg.get("prices"):
            print("  TCGPlayer (USD):")
            for cond, pd in tcg["prices"].items():
                m, lo, hi = pd.get("market"), pd.get("low"), pd.get("high")
                if any([m, lo, hi]):
                    lbl = cond.replace("Holofoil", "Holo").replace("ReverseHolofoil", "Rev.Holo")
                    print(f"    {lbl:<24} Market: {fmt(m,'$'):>9}   Low: {fmt(lo,'$'):>9}   High: {fmt(hi,'$'):>9}")
            if tcg.get("url"):
                print(f"    -> {tcg['url']}")

        cm = card.get("cardmarket", {})
        if cm and cm.get("prices"):
            p = cm["prices"]
            print("  Cardmarket (EUR):")
            print(f"    Trend: {fmt(p.get('trendPrice'),'EUR'):>10}   Avg-Verkauf: {fmt(p.get('averageSellPrice'),'EUR'):>10}   Avg7d: {fmt(p.get('avg7'),'EUR'):>10}   Avg30d: {fmt(p.get('avg30'),'EUR'):>10}")
            print(f"    Guenstigster: {fmt(p.get('lowPrice'),'EUR'):>10}   Guenstigster+: {fmt(p.get('lowPriceExPlus'),'EUR'):>10}")
            if cm.get("url"):
                print(f"    -> {cm['url']}")

    print(f"\n{SEP}")

In [ ]:
# ─── HIER BILDPFAD EINTRAGEN ───────────────────────────────────────────────
IMAGE_PATH = "card.jpg"   # Absoluter oder relativer Pfad zum Kartenfoto
# ───────────────────────────────────────────────────────────────────────────

# Optional: Bild anzeigen
from IPython.display import Image as IPImage
display(IPImage(filename=IMAGE_PATH, width=250))

In [ ]:
print(f"🔍 Analysiere Foto: {IMAGE_PATH}")
card_info = identify_card(IMAGE_PATH)

print(f"✅ Erkannt: {card_info.get('name')} | {card_info.get('set_name')} | #{card_info.get('card_number')}")
print(f"   Zustand: {card_info.get('condition_estimate')} | Holo: {card_info.get('is_holo')} | 1st Ed.: {card_info.get('is_first_edition')}")
print(f"\n(Vollständige Rohdaten für Debug:)")
print(json.dumps(card_info, indent=2, ensure_ascii=False))

In [ ]:
print("📊 Suche Marktpreise in Pokemon TCG API...")
cards = lookup_prices(card_info)
print(f"   {len(cards)} Treffer gefunden.")

In [ ]:
display_results(card_info, cards)

In [ ]:
# Manueller Override: Karte direkt per ID suchen (falls Vision-Erkennung falsch war)
# Beispiel: "base1-4" = Charizard Base Set
# Alle Set-IDs: https://pokemontcg.io/sets

MANUAL_CARD_ID = None  # z.B. "base1-4"

if MANUAL_CARD_ID:
    headers = {"X-Api-Key": POKEMON_TCG_API_KEY} if POKEMON_TCG_API_KEY else {}
    resp = requests.get(f"https://api.pokemontcg.io/v2/cards/{MANUAL_CARD_ID}", headers=headers)
    resp.raise_for_status()
    manual_card = resp.json().get("data", {})
    display_results(card_info, [manual_card])